# Copy vs View & Memory Sharing

## Definition
This is one of the most important (and most misunderstood) aspects of NumPy.

- **View**: A new array object that **shares memory** with the original. Changes to the view affect the original.
- **Copy**: An **independent** array with its own memory. Changes to the copy don't affect the original.

### When do you get a VIEW?
- Slicing: `arr[1:4]`, `arr[:, 0]`
- Reshaping (usually): `arr.reshape()`
- Transpose: `arr.T`
- `np.squeeze()`, `np.expand_dims()`
- `np.broadcast_to()`

### When do you get a COPY?
- Fancy indexing: `arr[[0, 2, 4]]`
- Boolean indexing: `arr[arr > 0]`
- `arr.copy()`
- `arr.flatten()` (NOT ravel)
- `np.concatenate()`
- Type casting with `astype()`

### How to check?
Use `arr.base` — if it's not `None`, the array is a view of that base.

In [ ]:
import numpy as np

# Slicing creates a VIEW
arr = np.array([1, 2, 3, 4, 5, 6])
view = arr[1:4]

print('Before:', arr)
print('View:', view)
view[0] = 99
print('After modifying view:', arr)  # arr[1] is now 99!
print()

# Check: view.base is arr
print('view.base is arr:', view.base is arr)   # True -> it's a view
print('arr.base:', arr.base)                    # None -> arr owns its data

In [ ]:
# Fancy indexing creates a COPY
arr = np.array([1, 2, 3, 4, 5, 6])
copy = arr[[0, 2, 4]]     # fancy indexing

print('Before:', arr)
copy[0] = 99
print('After modifying fancy-indexed copy:', arr)  # UNCHANGED
print('copy.base:', copy.base)   # None -> copy owns its data

In [ ]:
# reshape — usually a view
arr = np.arange(6)
reshaped = arr.reshape(2, 3)
reshaped[0, 0] = 999
print('After modifying reshaped, original:', arr)  # arr[0] changed!
print('reshaped.base is arr:', reshaped.base is arr)  # True

In [ ]:
# T (transpose) — always a view
arr = np.array([[1, 2, 3], [4, 5, 6]])
t = arr.T
t[0, 0] = 999
print('After modifying T, original:', arr)  # arr[0,0] changed!

In [ ]:
# Making an explicit copy
arr = np.array([1, 2, 3, 4, 5])
explicit_copy = arr.copy()
explicit_copy[0] = 999
print('After modifying copy, original:', arr)  # UNCHANGED

# Also: np.array(arr) makes a copy by default
copy2 = np.array(arr)
print('np.array() also copies:', copy2.base is None)  # True

In [ ]:
# Checking with np.shares_memory
arr = np.arange(6)
view = arr[::2]
copy = arr.copy()

print('shares_memory(arr, view):', np.shares_memory(arr, view))  # True
print('shares_memory(arr, copy):', np.shares_memory(arr, copy))  # False

In [ ]:
# Memory layout and strides
arr = np.array([[1, 2, 3],
                [4, 5, 6]])

# C-order (row-major) strides
print('C-order strides:', arr.strides)        # (24, 8) for int64

# F-order strides
arr_f = np.asfortranarray(arr)
print('F-order strides:', arr_f.strides)      # (8, 16) for int64

# transpose reverses strides
print('Transposed strides:', arr.T.strides)

In [ ]:
# When to use copy vs view in practice
import numpy as np

# Scenario: preprocess a batch without modifying original dataset
dataset = np.random.randint(0, 255, size=(1000, 784))  # 1000 images

# WRONG: modifying a slice modifies original
batch_view = dataset[:32]       # view!
batch_view /= 255.0              # modifies dataset!
print('Dataset dtype after view op:', dataset.dtype)

# CORRECT: work on a copy
dataset = np.random.randint(0, 255, size=(1000, 784))
batch_copy = dataset[:32].copy()  # explicit copy
batch_copy = batch_copy / 255.0   # safe, doesn't touch dataset
print('Dataset max still 254:', dataset.max() <= 255)